# 01 - Exploracion de datos

Prediccion de churn - Telco Customer Churn (IBM).

# Predicción de churn — Telco

## Problema

Los clientes cancelan el servicio y la empresa se entera cuando ya se fueron.
No hay forma de saber quién está en riesgo mientras todavía se puede hacer algo.
Se necesita identificarlos con anticipación para poder actuar a tiempo.

## Para quién

El área comercial, que ejecuta las acciones de retención sobre los clientes marcados.
Gerencia, que necesita ver la magnitud del riesgo para decidir presupuesto de retención.

## Qué se hace con el resultado

Comercial recibe la lista de clientes en riesgo y los contacta por teléfono para
ofrecer alternativas de plan o beneficios. En casos de alto valor se evalúa visita presencial.

Contactar a un cliente es barato. Perderlo es caro.
Por eso el modelo debe marcar de más antes que dejar pasar a alguien que sí se va.

## Cómo sabemos que quedó bien

Del modelo: que capture la mayor parte de los clientes que efectivamente se van,
sin marcar a tanta gente que la lista sea inmanejable para comercial.

Del negocio: que la tasa de cancelación mensual baje después de implementar el programa.

## Supuestos

- Contactar a un cliente en riesgo aumenta la probabilidad de retenerlo.
- Comercial tiene capacidad de atender el volumen de clientes que el modelo marque.
- El comportamiento histórico de los clientes se parece al de los clientes actuales.
- Los datos disponibles reflejan la situación del cliente antes de cancelar,
  no después de la decisión.

## Diccionario de datos

**Identificador**
- `customerID` — código único del cliente. No se usa para modelar.

**Demográficas**
- `gender` — Male / Female
- `SeniorCitizen` — 1 si tiene 65 años o más, 0 si no
- `Partner` — si tiene pareja (Yes/No)
- `Dependents` — si tiene personas a cargo (Yes/No)

**Servicios**
- `PhoneService` — si tiene servicio telefónico (Yes/No)
- `MultipleLines` — Yes / No / No phone service
- `InternetService` — DSL / Fiber optic / No
- `OnlineSecurity` — Yes / No / No internet service
- `OnlineBackup` — Yes / No / No internet service
- `DeviceProtection` — Yes / No / No internet service
- `TechSupport` — Yes / No / No internet service
- `StreamingTV` — Yes / No / No internet service
- `StreamingMovies` — Yes / No / No internet service

**Cuenta y facturación**
- `tenure` — meses que lleva el cliente con la empresa
- `Contract` — Month-to-month / One year / Two year
- `PaperlessBilling` — factura electrónica (Yes/No)
- `PaymentMethod` — Electronic check / Mailed check / Bank transfer (automatic) / Credit card (automatic)
- `MonthlyCharges` — cargo mensual actual
- `TotalCharges` — total facturado desde que es cliente

**Objetivo**
- `Churn` — Yes si canceló el servicio, No si sigue activo

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.info()

(7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    


In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].isnull().sum())

11


In [4]:
df[df['TotalCharges'].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [5]:
df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [6]:
df['OnlineSecurity'].value_counts()

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

In [7]:
df[df['TotalCharges'].isnull()]['Churn'].value_counts()

Churn
No    11
Name: count, dtype: int64

In [8]:
df = df[df['tenure'] > 0]

In [9]:
print(df.shape)

(7032, 21)


In [10]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True))

Churn
No     5163
Yes    1869
Name: count, dtype: int64
Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64


In [11]:
cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
        'TechSupport', 'StreamingTV', 'StreamingMovies']
df[cols] = df[cols].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

df['OnlineSecurity'].value_counts()

OnlineSecurity
No     5017
Yes    2015
Name: count, dtype: int64

In [12]:
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()
df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean()

,tenure,MonthlyCharges,TotalCharges
Churn,,,
No,37.650010,61.307408,2555.344141
Yes,17.979133,74.441332,1531.796094


Sobre tenure (1): tu lectura tiene lógica pero es una hipótesis, no lo que muestra el dato. El dato dice que se van temprano, no por qué. Mala conectividad, mal servicio, precio — todo eso es posible y no lo puedes saber con estas columnas.

Lo que sí puedes afirmar: el riesgo se concentra en los primeros meses. Y eso solo ya es accionable para comercial: el esfuerzo de retención va a los clientes nuevos, no a los de tres años.

Te dejo una pista para más adelante: hay una columna que probablemente explica mejor esa historia que la conectividad. Piensa qué tipo de contrato tiene alguien que puede irse a los 18 meses.

Sobre MonthlyCharges (2): vas bien con "planes más costosos". Falta el nombre concreto. Míralo:

In [13]:
df.groupby('InternetService')['MonthlyCharges'].mean()
df.groupby('InternetService')['Churn'].value_counts(normalize=True)

InternetService  Churn
DSL              No       0.810017
                 Yes      0.189983
Fiber optic      No       0.581072
                 Yes      0.418928
No               No       0.925658
                 Yes      0.074342
Name: proportion, dtype: float64